<a href="https://colab.research.google.com/github/project-ccap/project-ccap.github.io/blob/master/2025notebooks/2025_0717pmsp96_CDP_EncDec_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 準備作業

<img src="https://raw.githubusercontent.com/project-ccap/project-ccap.github.io/refs/heads/master/2025figs/1998Zorzi_CDP_fig1.svg" style="width:49%;"><br/>
<p>Zorzi+(1998) Fig.1 Architecture of the model. The arrow means full connectivity between layers. Each box stand for a group of letters (26) or phonemes (44).</p>


<img src="https://raw.githubusercontent.com/project-ccap/project-ccap.github.io/refs/heads/master/2025figs/1998Zorzi_CDP_fig8.svg" width="49%;"><br/>
<p>Zorzi+(1998) Fig.8. Architecture of the model with the hidden layer pathway. In both the direct pathway and the mediated pathway the layers are fully connected (arrows).</p>

<img src="https://raw.githubusercontent.com/project-ccap/project-ccap.github.io/refs/heads/master/2025figs/1998Zorzi_fig10.svg" width="49%"><br/>
<p style="align-text:center">
Figure 10. Lexical and sublexical procedures in reading aloud, and their interaction in the phonological decision system, where the final phonological code is computed for articulation.
</p>


In [ ]:
%config InlineBackend.figure_format = 'retina'
import torch
#device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'device:{device}')
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

# 必要なライブラリの輸入
from collections import OrderedDict
import sys
import os
import numpy as np
import operator
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# 全モデル共通使用するライブラリの輸入
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

HOME = os.environ['HOME']

from IPython import get_ipython
isColab =  'google.colab' in str(get_ipython())
print(f'isColab:{isColab}')

try:
    import japanize_matplotlib
except ImportError:
    !pip install japanize_matplotlib
    import japanize_matplotlib

try:
    import ipynbname
except ImportError:
    !pip install ipynbname
    import ipynbname

# FILEPATH = str(ipynbname.path()).split('/')[-1]
# print(f'FILEPATH:{FILEPATH}')

try:
    import CDP_ja
except ImportError:
    !git clone https://github.com/ShinAsakawa/CDP_ja.git
    import CDP_ja

# PMSP96 データ PMSPdata.txt のダウンロードと読み込み

In [ ]:
# Plaut の PMSP データを読み込む
import IPython
isColab = True if 'google.colab' in str(IPython.get_ipython()) else False

import numpy as np
import os
import requests
from collections import OrderedDict

if isColab:
    HOME = '/content'
else:
    HOME = os.environ['HOME']
pmsp_url='https://www.cnbc.cmu.edu/~plaut/xerion/PMSPdata.txt'
# 上のデータは以下のような一行一データであり tsv ファイルである。
# 以下のような構造である
# `orth\tphon\ttype\tSim 1\t\tSim 2 (raw)\tSim 2 (sqrt)\tSim 3 (RT)\n`

if isColab:
    xerion_dir = 'CDP_ja'
else:
    xerion_dir = 'study/2022plaut_homepage/xerion'
pmsp_fname = 'PMSPdata.txt'
fname = os.path.join(HOME, xerion_dir, pmsp_fname)

# Plaut の URL から PMSPdata.txt が削除されているようなので下記コードは不要 2025_0605
# # もしファイルが存在しなかったら ダウンロードする
# if not os.path.exists(fname):
#     r = requests.get(pmsp_url)
#     with open(fname, 'w') as f:
#         total_length = int(r.headers.get('content-length'))
#         print('Downloading {0} - {1} bytes'.format(pmsp_fname, (total_length)))
#         f.write(r.content)

with open(fname, 'r') as f:
    a = f.readlines()

PMSP96_dic = {}
for i,l in enumerate(a[1:]):
    x = l.strip().split('\t')
    PMSP96_dic[i] = {'orth':x[0], 'phon':x[1], 'type':x[2], 'Sim1':np.float32(x[3]),
                     'Sim2_raw':np.float32(x[4]), 'Sim2_sqrt':np.float32(x[5]), 'Sim3_RT':np.float32(x[6])}

# orth_list =[v['orth'] for k, v in Z.items()]
# phon_list =[v['phon'] for k, v in Z.items()]

special_tokens = ['<PAD>', '<UNK>', '<SOW>', '<EOW>', '<CLS>']
orth_dic, phon_dic = {},{}
for k, v in PMSP96_dic.items():
    for o in v['orth']:
        if not o in orth_dic:
            orth_dic[o] = 1
        else:
            orth_dic[o] += 1
    phon = v['phon'][1:-1]  # '/' を取り除くため
    for p in phon:
        if not p in phon_dic:
            phon_dic[p] = 1
        else:
            phon_dic[p] += 1

phon_tokens = special_tokens + list(sorted(phon_dic.keys()))
orth_tokens = special_tokens + list(sorted(orth_dic.keys()))

class _Tokenizer:
    def __init__(self,tokens:list=phon_tokens):
        self.tokens = tokens

    def __len__(self):
        return len(self.tokens)

    def encode(self, X):
        ret = []
        X = X.replace('/','')
        for _x in X:
            if not _x in self.tokens:
                ret.append(self.tokens.index('<UNK>'))
            else:
                ret.append(self.tokens.index(_x))
        return ret

    def decode(self, ids):
        ret = []
        for idx in ids:
            if (0 <= idx) and (idx <= len(self.tokens)):
                ret.append(self.tokens[idx])
            else:
                ret.append(-1)
        return ret

    def __call__(self, X):
        return self.encode(X)

orth_tokenizer = _Tokenizer(tokens=orth_tokens)
phon_tokenizer = _Tokenizer(tokens=phon_tokens)

input_tokenizer, output_tokenizer = orth_tokenizer, phon_tokenizer

# print(orth_tokenizer.__len__(), orth_tokenizer.tokens)
# print(phon_tokenizer.__len__(), phon_tokenizer.tokens)
# word = 'word'
# print(orth_tokenizer(word), orth_tokenizer.decode(orth_tokenizer(word)))
# phons = 'Ak'
# print(phon_tokenizer(phons), phon_tokenizer.decode(phon_tokenizer(phons)))
# PMSP96_dic[1]

# PMSP96 Dataset の定義

In [ ]:
import torch

class PMSP96_Dataset(torch.utils.data.Dataset):
    '''ニューラルネットワークモデルに Psylex71 を学習させるための PyTorch 用データセットのクラス'''

    def __init__(self,
                 dic:dict=PMSP96_dic,
                 _input:str = 'orth',
                 _output:str = 'phon',
                 inplen_min:int = 2,   # 最短文字列長
                 inplen_max:int = 8,   # 最長文字列長
                 outlen_max:int = 8,
                 input_tokenizer=orth_tokenizer,  # gakushu_tokenizer,   # 入力データのトークナイザ
                 output_tokenizer=phon_tokenizer, # mora_tokenizer,     # 出力データのトークナイザ
                 special_tokens:list = ['<PAD>', '<UNK>', '<SOW>', '<EOW>', '<CLS>'],
                 device:str=device,
                 display:bool=True,
                 add_special_tokens:bool=True,
                 isColab:bool=False):

        super().__init__()
        self.dic = dic
        self.input_tokenizer = input_tokenizer
        self.output_tokenizer = output_tokenizer
        self.special_tokens = special_tokens
        self._input = _input
        self._output = _output

        inp_maxlen = 0
        out_maxlen = 0
        # self.inp_maxlen = inplen_max
        # self.out_maxlen = outlen_max
        for x, v in dic.items():
            inp_maxlen = len(v[_input]) if len(v[_input]) > inp_maxlen else inp_maxlen
            out_maxlen = len(v[_output]) if len(v[_output]) > out_maxlen else out_maxlen

        self.inp_maxlen = inp_maxlen + 2     # +2 するのは 語頭と語尾とに <SOW>, <EOW> を加えるため
        self.out_maxlen = out_maxlen + 2

        self.device = device

    def __len__(self):
        return len(self.dic)

    def __getitem__(self, idx):
        orth = self.dic[idx][self._input]
        phon = self.dic[idx][self._output]
        inp = [self.input_tokenizer.tokens.index('<SOW>')]+self.input_tokenizer(orth)+[self.input_tokenizer.tokens.index('<EOW>')]
        tch = [self.output_tokenizer.tokens.index('<SOW>')]+self.output_tokenizer(phon)+[self.output_tokenizer.tokens.index('<EOW>')]

        while len(inp) < self.inp_maxlen:
            inp = inp + [self.output_tokenizer.tokens.index('<PAD>')]
        while len(tch) < self.out_maxlen:
            tch = tch + [self.output_tokenizer.tokens.index('<PAD>')]

        inp, tch = torch.LongTensor(inp).to(self.device), torch.LongTensor(tch).to(device)
        return inp, tch

    def getitem(self, idx):
        orth = self.dic[idx][self._input]
        phon = self.dic[idx][self._output]
        return orth, phon

pmsp96_ds = PMSP96_Dataset()
pmsp96_ds.__len__()
idx = 0
print(pmsp96_ds.__getitem__(idx))
print(pmsp96_ds.getitem(idx))
print(pmsp96_ds.inp_maxlen, pmsp96_ds.out_maxlen)

# データセットの分割,訓練,検査,検証データセット

In [ ]:
# データセットの分割,訓練,検査,検証データセット

# 乱数の種を設定
seed=42

_ds = pmsp96_ds

# 訓練データセット,検証データセット,検査データセットに 3 分割するための割合を宣言
train_size = int(_ds.__len__() * 0.8)
valid_size = _ds.__len__() - train_size

# 実際のデータ分割
train_ds, valid_ds = torch.utils.data.random_split(
    dataset=_ds, lengths=(train_size, valid_size), generator=torch.Generator().manual_seed(seed))

# ミニバッチサイズ
batch_size = 128

# データセットとミニバッチサイズを用いて PyTorch 用のデータローダを宣言
train_dl = torch.utils.data.DataLoader(dataset=train_ds, batch_size=batch_size, shuffle=True)
valid_dl = torch.utils.data.DataLoader(dataset=valid_ds, batch_size=batch_size, shuffle=False)

# 並列計算のための準備
def _collate_fn(batch):
    inps, tgts = list(zip(*batch))
    inps = list(inps)
    tgts = list(tgts)
    return inps, tgts

# 訓練データセット用データローダ
train_dl = torch.utils.data.DataLoader(
    dataset=train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=_collate_fn)

# 検証データセット用のデータローダ
valid_dl = torch.utils.data.DataLoader(
    dataset=valid_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=_collate_fn)

print(f'train_ds.__len__():{train_ds.__len__()}')
print(f'valid_ds.__len__():{valid_ds.__len__()}')

# モデルの宣言


In [ ]:
%load_ext autoreload
%autoreload 2

from CDP_ja import direct_TLA, indirect_TLA, combined_TLA
from CDP_ja import Seq2Seq_wAtt, Seq2Seq_woAtt
from CDP_ja import fit_an_epoch
from CDP_ja import eval_an_epoch
from CDP_ja import Transformer
#from CDP_ja import Psylex71_Dataset

tla_direct = direct_TLA(
    inp_vocab_size=len(input_tokenizer.tokens),
    out_vocab_size=len(output_tokenizer.tokens),
    out_f='none',
    inp_len = pmsp96_ds.inp_maxlen,
    out_len = pmsp96_ds.out_maxlen,
    device=device)
print(tla_direct.eval())

tla_indirect = indirect_TLA(
    inp_vocab_size=len(input_tokenizer.tokens),
    out_vocab_size=len(output_tokenizer.tokens),
    out_f='none',
    inp_len = pmsp96_ds.inp_maxlen,
    out_len = pmsp96_ds.out_maxlen,
    device=device)
print(tla_indirect.eval())

tla_combined = combined_TLA(
    inp_vocab_size=len(input_tokenizer.tokens),
    out_vocab_size=len(output_tokenizer.tokens),
    out_f='none',
    inp_len = pmsp96_ds.inp_maxlen,
    out_len = pmsp96_ds.out_maxlen,
    device=device)
print(tla_combined.eval())

num_layers = 1
bidirectional = False
n_hid = 1024

seq2seq = Seq2Seq_wAtt(
    enc_vocab_size=len(input_tokenizer.tokens),
    dec_vocab_size=len(output_tokenizer.tokens),
    n_layers=num_layers,
    bidirectional=bidirectional,
    n_hid=n_hid).to(device)
print(seq2seq.eval())

seq2seqN = Seq2Seq_woAtt(
    enc_vocab_size=len(input_tokenizer.tokens),
    dec_vocab_size=len(output_tokenizer.tokens),
    n_layers=num_layers,
    bidirectional=bidirectional,
    n_hid=n_hid).to(device)
print(seq2seqN.eval())

transformer = Transformer(
    src_vocab_size=len(input_tokenizer.tokens),
    tgt_vocab_size=len(output_tokenizer.tokens),
    model_dim=n_hid,
    num_heads=4,
    num_layers=1,
    max_seq_length=pmsp96_ds.inp_maxlen if pmsp96_ds.inp_maxlen > pmsp96_ds.out_maxlen else pmsp96_ds.out_maxlen,
    dropout=0.,
    ff_dim=32,
    device=device)
print(transformer.eval())

inps, tchs = next(iter(train_dl))
inps = pad_sequence(inps, batch_first=True).to(device)
tchs = pad_sequence(tchs, batch_first=True).to(device)
print(inps.size(), tchs.size())
outs1 = tla_direct(inps, tchs)
outs2 = tla_indirect(inps, tchs)
outs3 = tla_combined(inps, tchs)
outs4, outs4_state = seq2seq(inps, tchs)
outs5, outs5_state = seq2seqN(inps, tchs)
#outs6 = transformer(inps, tchs)
#outs1.size(), outs2.size(), outs3.size(), outs4.size(), outs5.size(), outs6.size()

In [ ]:
model1 = tla_direct
model2 = tla_indirect
model3 = tla_combined
model4 = seq2seq
model5 = seq2seqN
model6 = transformer

models = [model1, model2, model3, model4, model5, model6]
results = [{} for _ in models]

#loss_f = torch.nn.CrossEntropyLoss(ignore_index=0)
loss_f = torch.nn.CrossEntropyLoss()

# 実際の訓練

In [ ]:
from CDP_ja import fit_an_epoch
from CDP_ja import eval_an_epoch

# 学習率リストとエポック数の定義
iter_params = [(1e-3, 2), (1e-4, 2), (1e-5, 2)]
iter_params = [(1e-3, 20)]
iter_params = [(1e-3, 5)]
iter_params = [(1e-2, 20)]
iter_params = [(1e-1, 50), (1e-2,20), (1e-3,20), (1e-4,10)]
iter_params = [(1e-3, 20)]
#iter_params = [(1e-1, 30)]
# iter_params = [(1e-4, 10)]
# iter_params = [(1e-2, 50), (1e-3, 50), (1e-4, 10)]

try:
    isinstance(results, list)
except:
    results = [{} for _ in models]

# 途中結果を印字するタイミング
#interval = 3
interval = 1
#interval = 10

for (lr, epochs) in iter_params: # 学習率とエポック数を定義済のリストに従って変化させる

    # 最適化関数の学習率を設定
    optimizers = []
    for model in models:
        optimizers.append(torch.optim.Adam(model.parameters(), lr=lr))

    print(f'lr:{lr}, epochs:{epochs}')
    # エポック数だけ学習を行う
    for epoch in range(epochs):
        print(f"エポック:{epoch+1:3d}")

        for N, (model, optimizer) in enumerate(zip(models, optimizers)):
            model_name = str(type(model)).split('\'')[1].split('.')[-1]

            # 1 エポックの検証を行う
            is_seq2seq = 'Seq2Seq' in str(type(model))
            out, model = eval_an_epoch(
                model=model,
                _dl=valid_dl,
                loss_f=loss_f,
                is_seq2seq=is_seq2seq,
                device=device)
            if (epoch % interval) == 0:
                print(f"{model_name:14s}",
                      f"検証損失値={out['sum_loss']:10.3f}",
                      f"正解率={out['P']:5.3f}",
                      f"({out['count']:5d}/{out['N']:5d})",
                      end="\t")

            if not 'valid_loss' in results[N]:
                results[N]['valid_loss'] = [out['sum_loss']]
            else:
                results[N]['valid_loss'].append(out['sum_loss'])
            if not 'valid_P' in results[N]:
                results[N]['valid_P'] = [out['P']]
            else:
                results[N]['valid_P'].append(out['P'])


            # 1 エポックの訓練を行う
            out, model, optimizer = fit_an_epoch(
                model=model,
                _dl=train_dl,
                loss_f=loss_f,
                optimizer=optimizer,
                is_seq2seq=is_seq2seq,
                device=device)
            if (epoch % interval) == 0:
                print(f"学習損失値={out['sum_loss']:10.3f}",
                      f"正解率={out['P']:5.3f}",
                      f"({out['count']:5d}/{out['N']:5d})",
                      end="\t")

            if not 'train_loss' in results[N]:
                results[N]['train_loss'] = [out['sum_loss']]
            else:
                results[N]['train_loss'].append(out['sum_loss'])
            if not 'train_P' in results[N]:
                results[N]['train_P'] = [out['P']]
            else:
                results[N]['train_P'].append(out['P'])

            if (epoch % interval) == 0:
                print()

In [ ]:
_ds = pmsp96_ds

_errors = {}
for N, model in enumerate(models[:]):
    model = model.eval()
    model_name = str(type(model)).split('\'')[1].split('.')[-1]

    if not model_name in _errors:
        _errors[model_name] = []

    is_seq2seq = 'seq2seq' in str(type(model)).lower()
    verbose = False
    errors = []
    for idx in tqdm(range(_ds.__len__() )):
        #_ds.getitem(idx), _ds.__getitem__(idx)
        inp, tch = _ds.__getitem__(idx)
        inp = pad_sequence(inp.unsqueeze(0), batch_first=True).to(device)
        tch = pad_sequence(tch.unsqueeze(0), batch_first=True).to(device)
        if is_seq2seq:
            enc_inp, enc_tch = inp[:,:-1], inp[:,1:]
            dec_inp, dec_tch = tch[:,:-1], tch[:,1:]
            dec_out, enc_out = model(enc_inp, dec_inp)
            out = dec_out.squeeze(0).argmax(dim=1)
            tch = dec_tch.squeeze(0)
        else:
            out = model(inp,tch)
            out = out.squeeze(0).argmax(dim=1)
            tch = tch.squeeze(0)

        yesno = (((out == tch) * 1).sum() == len(tch)).detach().cpu().numpy()
        if yesno != True:
            if verbose:
                print(f'{idx:07d}',
                      f'{yesno}',
                      f'出力:{"".join(ch for ch in _ds.output_tokenizer.decode(out)).replace("<PAD>","")}',
                      f'{_ds.getitem(N)}')
            else:
                _errors[model_name].append((f'{idx:07d}',
                                            f'{yesno}',
                                            f'出力:{"".join(ch for ch in _ds.output_tokenizer.decode(out)).replace("<PAD>","")}',
                                            f'{_ds.getitem(idx)}'))

    print(f'{model_name}',
      f'total num. erros:{len(_errors[model_name])}', '/', f'{_ds.__len__()}',
      f'P(Correct)={((_ds.__len__() - len(_errors[model_name])) / _ds.__len__())*100:.3f}%')

In [ ]:
for i in range(len(models)):

    # plt.plot(results[i]['train_loss'], 'x-', label=f'{i}:訓練データ')
    # plt.plot(results[i]['valid_loss'], 'o-', label=f'{i}:検証データ')
    #plt.plot(results[i]['train_P'], 'x-', label=f'モデル{i}:訓練データ')
    plt.plot(results[i]['valid_P'], 'o-', label=f'モデル{i}:検証データ')

plt.legend()
plt.xlabel('エポック数')
#plt.ylim(0,1)
plt.show()

#print(results)


In [ ]:
tla_direct.eval()
for idx in range(_ds.__len__() >> 3):
        #_ds.getitem(idx), _ds.__getitem__(idx)
        inp, tch = _ds.__getitem__(idx)
        inp = pad_sequence(inp.unsqueeze(0), batch_first=True).to(device)
        tch = pad_sequence(tch.unsqueeze(0), batch_first=True).to(device)
        if is_seq2seq:
            out, state = model(inp,tch)
        else:
            out = model(inp,tch)
        out = out.squeeze(0).argmax(dim=1)
        tch = tch.squeeze(0)
        yesno = (((out == tch) * 1).sum() == len(tch)).detach().cpu().numpy()




In [ ]:
import matplotlib.pyplot as plt
try:
    import japanize_matplotlib
except:
    !pip install japanize_matplotlib
    import japanize_matplotlib

for i in range(len(results)):
    #plt.plot(results[i]['train_loss'], 'x-', label=f'{i}:訓練データ')
    #plt.plot(results[i]['valid_loss'], 'o-', label=f'{i}:検証データ')
    plt.plot(results[i]['train_P'], 'x-', label=f'モデル{i}:訓練データ')
    plt.plot(results[i]['valid_P'], 'o-', label=f'モデル{i}:検証データ')

plt.legend()
plt.xlabel('エポック数')
plt.show()

print(results)
